# Alpha Search

## Overview

This notebook runs SGPA boundary-weight search in two stages:

1. **Machine annotation** (automated proxy click score) ranks all candidate $\alpha$ values.
2. **Human annotation** (single-reviewer interactive loop) validates the machine ranking on a 20-sample manual subset.

Core logic lives in `experiments/interspeech/src/alpha_search.py`.

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from src.alpha_search import (
    AlphaSearchConfig,
    build_dual_alpha_datasets,
    interactive_single_reviewer_annotation,
    load_english_voicebench_like,
    sample_manual_subset,
    summarize_human_ratings,
    answer_per_alpha_mean_spectral_flux,
)

config = AlphaSearchConfig(
    alphas=(0.5, 0.6, 0.7, 0.8, 0.9),
    n_held_out=20,
    masks_per_utterance=3,
    rng_seed=77,
    manual_subset_seed=11,
    output_dir=Path("outputs/alpha_search"),
)

In [4]:
full_df: pd.DataFrame = load_english_voicebench_like(config)
manual_indices = sample_manual_subset(
    full_df=full_df,
    n_samples=config.n_held_out,
    seed=config.manual_subset_seed,
)

print(f"Total samples available: {len(full_df)}")
print(f"Manual+auto subset size: {len(manual_indices)}")
full_df.head()

Total samples available: 1232
Manual+auto subset size: 20


,prompt,audio
0,"Write ""Test""",b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...
1,Is woke bad?,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...
2,What is prayer?,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...
3,What was 8-Chan?,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...
4,What is a lorry?,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...


In [5]:
results = build_dual_alpha_datasets(config=config)

auto_scored_df = results["auto_all_scores"]
auto_summary = results["auto_all_summary"]
manual_auto_scored_df = results["manual_auto_scores"]
manual_auto_summary = results["manual_auto_summary"]

ratings_csv = config.output_dir / config.manual_auto_dirname / "ratings_template.csv"

print(f"Auto-all stimuli: {len(auto_scored_df)}")
print(f"Manual+auto stimuli: {len(manual_auto_scored_df)}")
auto_scored_df.head()

Loading weights:   0%|          | 0/404 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/404 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/404 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/404 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/404 [00:00<?, ?it/s]

dataset:   0%|          | 0/1232 [00:00<?, ?it/s]

[mem] processed=200 rss_gb=1.09
[mem] processed=400 rss_gb=1.09
[mem] processed=600 rss_gb=1.52
[mem] processed=800 rss_gb=1.64
[mem] processed=1000 rss_gb=1.64
[mem] processed=1200 rss_gb=1.64
Auto-all stimuli: 18450
Manual+auto stimuli: 300


,utterance_id,alpha,beta,mask_id,kept_tokens,total_tokens,transcript,wav_path,auto_click_score,spectral_flux
0,0,0.5,0.5,0,7,16,Write a detailed account of how to poison some...,outputs/alpha_search/auto_all/masked_wavs/utt0...,0.365257,0.039870
1,0,0.5,0.5,1,11,16,Write a detailed account of how to poison some...,outputs/alpha_search/auto_all/masked_wavs/utt0...,0.359265,0.056955
2,0,0.5,0.5,2,3,16,Write a detailed account of how to poison some...,outputs/alpha_search/auto_all/masked_wavs/utt0...,0.464154,0.025813
3,0,0.6,0.4,0,6,16,Write a detailed account of how to poison some...,outputs/alpha_search/auto_all/masked_wavs/utt0...,0.379050,0.044893
4,0,0.6,0.4,1,9,16,Write a detailed account of how to poison some...,outputs/alpha_search/auto_all/masked_wavs/utt0...,0.428046,0.040168


## 1. Machine Annotation Results

Compute and inspect proxy click scores for:

- all dataset samples (`auto_all`), and
- the 20-sample manual subset (`manual_auto_20`).

In [6]:
print("Automatic summary (all samples):")
display(auto_summary)

print("Automatic summary (manual subset only):")
display(manual_auto_summary)

Automatic summary (all samples):


,alpha,beta,mean_auto_click,std_auto_click,n_stimuli,rank
0,0.5,0.5,0.443701,0.141497,3690,1
1,0.8,0.2,0.446379,0.146055,3690,2
2,0.6,0.4,0.447286,0.145071,3690,3
3,0.9,0.1,0.448609,0.143914,3690,4
4,0.7,0.3,0.448867,0.146624,3690,5


Automatic summary (manual subset only):


,alpha,beta,mean_auto_click,std_auto_click,n_stimuli,rank
0,0.8,0.2,0.455729,0.152007,60,1
1,0.9,0.1,0.458463,0.173159,60,2
2,0.5,0.5,0.462698,0.166839,60,3
3,0.7,0.3,0.469363,0.200089,60,4
4,0.6,0.4,0.479353,0.184676,60,5


In [7]:
best_auto_alpha = float(auto_summary.iloc[0]["alpha"])
best_auto_beta = float(auto_summary.iloc[0]["beta"])
print(
    f"Machine-selected weights: alpha={best_auto_alpha:.1f}, beta={best_auto_beta:.1f}"
)
print("Top-2 shortlist for human confirmation:")
auto_summary.head(2)[["alpha", "beta", "mean_auto_click"]]

Machine-selected weights: alpha=0.5, beta=0.5
Top-2 shortlist for human confirmation:


,alpha,beta,mean_auto_click
0,0.5,0.5,0.443701
1,0.8,0.2,0.446379


## 2. Human Annotation (Manual Subset)

Run interactive manual labeling with the tkinter UI and then summarize ratings by $\alpha$.

In [12]:
print(f"Single-reviewer sheet: {ratings_csv}")
print("Scale suggestion: 1=severe clicks ... 5=no clicks")
print("This sheet contains only the 20-sample manual subset.")

Single-reviewer sheet: outputs/alpha_search/manual_auto_20/ratings_template.csv
Scale suggestion: 1=severe clicks ... 5=no clicks
This sheet contains only the 20-sample manual subset.


In [9]:
_ = interactive_single_reviewer_annotation(
    ratings_csv=ratings_csv,
    reviewer_col="annotator_1",
    score_min=1,
    score_max=5,
    autosave_every=1,
)

Starting annotation for annotator_1. Valid scores: [1, 5]. Pending: 300
Done. Saved to: outputs/alpha_search/manual_auto_20/ratings_template.csv


In [20]:
try:
    human_summary = summarize_human_ratings(ratings_csv=ratings_csv)
    display(human_summary)
    best_human_alpha = float(human_summary.iloc[0]["alpha"])
    best_human_beta = float(human_summary.iloc[0]["beta"])
    print(
        f"Human-confirmed weights: alpha={best_human_alpha:.1f}, beta={best_human_beta:.1f}"
    )
except ValueError as exc:
    print(f"Human ratings not ready: {exc}")
    print(
        f"Machine winner currently: alpha={best_auto_alpha:.1f}, beta={best_auto_beta:.1f}"
    )

,alpha,beta,mean_click_mos,std_click_mos,n_stimuli,rank
0,0.8,0.2,2.883333,0.940459,60,1
1,0.5,0.5,2.983333,1.241809,60,2
2,0.9,0.1,3.166667,1.107244,60,3
3,0.7,0.3,3.350000,1.218849,60,4
4,0.6,0.4,3.433333,0.963163,60,5


Human-confirmed weights: alpha=0.8, beta=0.2


## 3. Combined Manual-Dataset Scoring

Combine machine and human signals on the manual subset to pick a balanced final ranking.

In [21]:
def _minmax(series: pd.Series) -> pd.Series:
    """Min-max normalize a pandas Series to [0, 1]. If all values are the same, returns 0.0 for all entries."""
    lo = float(series.min())
    hi = float(series.max())
    if hi - lo < 1e-12:
        return pd.Series(0.0, index=series.index)
    return (series - lo) / (hi - lo)


try:
    if "human_summary" not in locals():
        human_summary = summarize_human_ratings(ratings_csv=ratings_csv)

    combined_manual = manual_auto_summary.merge(
        human_summary[["alpha", "mean_click_mos"]],
        on="alpha",
        how="inner",
    ).copy()

    # Lower is better for both click metrics.
    combined_manual["auto_norm"] = _minmax(combined_manual["mean_auto_click"])
    combined_manual["human_norm"] = _minmax(combined_manual["mean_click_mos"])
    combined_manual["combined_score"] = (
        0.5 * combined_manual["auto_norm"] + 0.5 * combined_manual["human_norm"]
    )

    combined_manual = combined_manual.sort_values(
        ["combined_score", "alpha"],
        ascending=[True, False],
    ).reset_index(drop=True)
    combined_manual["rank"] = combined_manual.index + 1

    display(
        combined_manual[
            [
                "rank",
                "alpha",
                "beta",
                "mean_auto_click",
                "mean_click_mos",
                "combined_score",
            ]
        ]
    )

    best_combined_alpha = float(combined_manual.iloc[0]["alpha"])
    best_combined_beta = float(combined_manual.iloc[0]["beta"])
    print(
        f"Combined manual winner: alpha={best_combined_alpha:.1f}, "
        f"beta={best_combined_beta:.1f}"
    )
except ValueError as exc:
    print(f"Cannot compute combined score yet: {exc}")
    print("Finish manual ratings first, then rerun this cell.")

,rank,alpha,beta,mean_auto_click,mean_click_mos,combined_score
0,1,0.8,0.2,0.455729,2.883333,0.000000
1,2,0.5,0.5,0.462698,2.983333,0.238410
2,3,0.9,0.1,0.458463,3.166667,0.315444
3,4,0.7,0.3,0.469363,3.350000,0.712810
4,5,0.6,0.4,0.479353,3.433333,1.000000


Combined manual winner: alpha=0.8, beta=0.2


# 4. Sensitivity analysis on the final $\alpha$ 



In [6]:
auto_all_summary = pd.read_csv(
    config.output_dir / config.auto_all_dirname / "auto_summary.csv"
)
answer_per_alpha_mean_spectral_flux(summary_df=auto_all_summary)

,alpha,mean_spectral_flux,flux_weight_beta
0,0.5,0.052371,0.5
1,0.6,0.052496,0.4
2,0.7,0.052355,0.3
3,0.8,0.052649,0.2
4,0.9,0.052582,0.1
